# 08 An autoencoder on expression-like data

Lecture 11, architectures and representations. Book chapter 11 (autoencoders
and representation learning).

**Goals**

- Simulate expression-like data whose cells lie on a curved, branching
  trajectory in a 2-D latent space (`expression.py` in this folder).
- Train an autoencoder with a 2-D bottleneck and compare its reconstruction
  error with PCA and kernel PCA (lectures 1 and 5).
- Look at the 2-D representations that the three methods learn.

**Expected finding** (asserted in
`tests/test_examples.py::test_08_*`): training reduces the test
reconstruction error of the autoencoder more than threefold; with a 2-D
code it reconstructs at least 20 % better than 2-D PCA and better than 3-D
PCA, while 2-D kernel PCA (with its approximate pre-image) reconstructs
worse than 2-D PCA.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from expression import CELL_TYPES, make_expression_data
from sklearn.decomposition import PCA, KernelPCA
from sklearn.neighbors import KNeighborsClassifier

import glassnn.functional as F
from glassnn import Tensor, manual_seed, nn, no_grad, optim
from glassnn.data import DataLoader

X_train_raw, z_train, type_train = make_expression_data(1000, seed=0)
X_test_raw, z_test, type_test = make_expression_data(1000, seed=10)
mean, std = X_train_raw.mean(axis=0), X_train_raw.std(axis=0)
X_train, X_test = (X_train_raw - mean) / std, (X_test_raw - mean) / std  # per gene


def mse(a, b):
    return float(((a - b) ** 2).mean())


fig, ax = plt.subplots(figsize=(5, 3))
for t, name in enumerate(CELL_TYPES):
    ax.scatter(*z_train[type_train == t].T, s=4, label=name)
ax.set_title("true latent positions")
ax.legend(markerscale=3, fontsize=8)
plt.show()
print("expression matrix:", X_train.shape)

## The autoencoder

Encoder $200 \to 64 \to 2$ and decoder $2 \to 64 \to 200$ with GELU, trained
to reproduce its input under the mean squared error. The 2-D bottleneck
forces it to summarize every cell by two numbers.

In [ ]:
manual_seed(0)
encoder = nn.Sequential(nn.Linear(200, 64), nn.GELU(), nn.Linear(64, 2))
decoder = nn.Sequential(nn.Linear(2, 64), nn.GELU(), nn.Linear(64, 200))
autoencoder = nn.Sequential(encoder, decoder)
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)

history = []
for _ in range(60):
    for xb, _ in DataLoader(
        X_train, np.zeros(len(X_train)), batch_size=64, shuffle=True
    ):
        optimizer.zero_grad()
        F.mse_loss(autoencoder(xb), xb).backward()
        optimizer.step()
    with no_grad():
        history.append(
            (
                mse(X_train, autoencoder(Tensor(X_train)).data),
                mse(X_test, autoencoder(Tensor(X_test)).data),
            )
        )
history = np.array(history)
plt.figure(figsize=(5, 3))
plt.plot(history[:, 0], label="train")
plt.plot(history[:, 1], label="test")
plt.xlabel("epoch")
plt.ylabel("reconstruction MSE")
plt.legend()
plt.show()

## Comparison with PCA and kernel PCA

PCA with $k$ components reconstructs by projecting onto the best
$k$-dimensional *linear* subspace. Kernel PCA finds nonlinear components,
but has no exact way back from the components to the genes; scikit-learn
learns an approximate pre-image map (`fit_inverse_transform=True`). The
kernel width $\gamma$ is chosen from a small grid by the test error, which
favours kernel PCA a little.

In [ ]:
errors = {}
for k in [1, 2, 3, 4, 5, 10]:
    pca = PCA(k).fit(X_train)
    errors[f"PCA {k}"] = mse(X_test, pca.inverse_transform(pca.transform(X_test)))
kernel_pcas = {
    gamma: KernelPCA(
        2, kernel="rbf", gamma=gamma, fit_inverse_transform=True, alpha=0.1
    ).fit(X_train)
    for gamma in (0.001, 0.003, 0.01)
}
kernel_errors = {
    gamma: mse(X_test, kp.inverse_transform(kp.transform(X_test)))
    for gamma, kp in kernel_pcas.items()
}
best_gamma = min(kernel_errors, key=kernel_errors.get)
errors[f"kernel PCA 2 (gamma {best_gamma})"] = kernel_errors[best_gamma]
with no_grad():
    errors["autoencoder 2"] = mse(X_test, autoencoder(Tensor(X_test)).data)
for name, value in errors.items():
    print(f"{name:28s} test MSE {value:.3f}")

## The 2-D representations

Each method maps a cell to two numbers. How well do they keep the cell type?
A 15-nearest-neighbour classifier, trained on the training cells and tested
on the test cells, measures it.

In [ ]:
pca2 = PCA(2).fit(X_train)
with no_grad():
    codes = {
        "PCA": (pca2.transform(X_train), pca2.transform(X_test)),
        "kernel PCA": (
            kernel_pcas[best_gamma].transform(X_train),
            kernel_pcas[best_gamma].transform(X_test),
        ),
        "autoencoder": (encoder(Tensor(X_train)).data, encoder(Tensor(X_test)).data),
    }
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, (train_code, test_code)) in zip(axes, codes.items(), strict=True):
    knn = KNeighborsClassifier(15).fit(train_code, type_train)
    for t, label in enumerate(CELL_TYPES):
        ax.scatter(*test_code[type_test == t].T, s=4, label=label)
    ax.set_title(f"{name}: kNN accuracy {knn.score(test_code, type_test):.2f}")
axes[0].legend(markerscale=3, fontsize=8)
plt.show()

**Findings.**

1. The autoencoder with a 2-D code reconstructs about as well as PCA with 4
   to 5 components, and clearly better than 2-D PCA: the data lie near a
   2-D *curved* surface, which a linear projection can follow only with
   extra dimensions.
2. Kernel PCA finds nonlinear components too, but its approximate pre-image
   reconstructs worse than plain PCA; the autoencoder learns the decoder
   together with the encoder.
3. For the cell types, all three 2-D representations do about equally well
   here (kNN accuracy about 0.9); a better reconstruction does not
   automatically give a better representation for a given downstream task.

**Exercises.**

1. Remove the GELU activations. The autoencoder is then linear; compare its
   error and its 2-D code with PCA. What does it learn?
2. Use a 1-D bottleneck. Can a single number describe a branching
   trajectory? Look at where the two branches go.
3. Train a *denoising* autoencoder: feed $X + \varepsilon$ with Gaussian
   noise, reconstruct $X$. How do the test error and the code change?